# Actualización diaria del CSV de criptomonedas

Notebook pensado para uso diario: lo corres entero cada día, descarga el día/días
que falten, calcula dominancias **reales** (no extrapoladas), hace backup del CSV
anterior y guarda el actualizado.

## Lo que hace cada vez que se ejecuta

1. Carga `df_merged.csv` y detecta cuántos días faltan.
2. Si está al día, sale sin hacer nada (no toca el CSV).
3. Si faltan días, descarga de CoinGecko: precio + volumen + market cap de BTC y ETH.
4. Descarga de CoinGecko `/global`: market cap total **del día actual** (real, gratis).
5. Calcula las 3 dominancias para los días nuevos.
6. Descarga Fear & Greed actualizado.
7. Mergea, hace backup, guarda.

## Decisión de diseño honesta

El endpoint `/global` solo da el market cap total del **momento actual**, no histórico.
Si solo añades 1-2 días, asumimos que el market cap total del último día es válido para
calcular dominancias del día anterior también. El error es despreciable (<0.5%).

Si dejas pasar más de 7 días, el script avisa para que descargues el CSV manual de
CoinGecko Global Charts y lo mergees con el otro notebook (`mergearcsv.ipynb`).

## 1. Configuración

In [1]:
import os
import time
import shutil
import requests
import pandas as pd
from datetime import datetime

# Rutas
BASE_DIR    = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"
RUTA_CSV    = os.path.join(BASE_DIR, "data", "csv", "df_merged.csv")
DIR_BACKUPS = os.path.join(BASE_DIR, "data", "csv", "backups")

# APIs
COINGECKO_BASE   = "https://api.coingecko.com/api/v3"
ALTERNATIVE_BASE = "https://api.alternative.me/fng/"

# Rate limit (free tier ~10-30 calls/min, vamos conservadores)
PAUSA_API = 2.5

# Si faltan más días que esto, el script avisa para usar mergearcsv.ipynb
MAX_DIAS_AUTO = 7

os.makedirs(DIR_BACKUPS, exist_ok=True)

print("Configuración cargada ✓")
print(f"  CSV principal: {RUTA_CSV}")
print(f"  Backups en   : {DIR_BACKUPS}")

Configuración cargada ✓
  CSV principal: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv
  Backups en   : C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\backups


## 2. Diagnóstico

In [2]:
df_merged = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df_merged.sort_index(inplace=True)

ultima_fecha = df_merged.index.max().normalize()
hoy          = pd.Timestamp.now().normalize()
dias_faltan  = (hoy - ultima_fecha).days

print(f"Última fecha en CSV : {ultima_fecha.date()}")
print(f"Hoy                 : {hoy.date()}")
print(f"Días a actualizar   : {dias_faltan}")
print(f"Total filas actuales: {len(df_merged)}")

if dias_faltan <= 0:
    print("\n✓ El CSV ya está al día. Si ejecutas el resto, no hará nada.")
elif dias_faltan > MAX_DIAS_AUTO:
    print(f"\n⚠️  Faltan {dias_faltan} días, más del umbral seguro de {MAX_DIAS_AUTO}.")
    print("   Para huecos grandes, usa mergearcsv.ipynb con el CSV manual de")
    print("   CoinGecko Global Charts para evitar extrapolar dominancias.")
else:
    print(f"\n→ Procediendo a descargar {dias_faltan} día(s) nuevo(s).")

Última fecha en CSV : 2026-05-06
Hoy                 : 2026-05-07
Días a actualizar   : 1
Total filas actuales: 3017

→ Procediendo a descargar 1 día(s) nuevo(s).


## 3. Funciones de descarga (con reintento)

In [3]:
def get_con_reintento(url, params=None, intentos=3, espera=5):
    """GET con reintentos si la API tiene rate limit u otro 5xx."""
    for n in range(1, intentos + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 502, 503, 504):
                print(f"  ⚠️  {r.status_code} - reintento {n}/{intentos} en {espera}s")
                time.sleep(espera)
                continue
            raise RuntimeError(f"Error {r.status_code}: {r.text[:200]}")
        except requests.exceptions.RequestException as e:
            if n == intentos:
                raise
            print(f"  ⚠️  excepción - reintento {n}/{intentos}: {e}")
            time.sleep(espera)
    raise RuntimeError("Falló tras reintentos")


def ajustar_dias_coingecko(dias):
    """CoinGecko free solo acepta valores discretos."""
    for v in [1, 7, 14, 30, 90, 180, 365]:
        if dias <= v:
            return v
    return 365


def descargar_market_chart(coin_id, dias):
    """Descarga close, volume, market_cap diarios. Devuelve DataFrame indexado."""
    url = f"{COINGECKO_BASE}/coins/{coin_id}/market_chart"
    data = get_con_reintento(url, {"vs_currency": "usd", "days": dias, "interval": "daily"})
    
    df_p = pd.DataFrame(data["prices"],        columns=["ts", "close"])
    df_v = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])
    df_m = pd.DataFrame(data["market_caps"],   columns=["ts", "market_cap"])
    
    for d in (df_p, df_v, df_m):
        d["date"] = pd.to_datetime(d["ts"], unit="ms").dt.normalize()
        d.drop(columns="ts", inplace=True)
        d.set_index("date", inplace=True)
    
    df = df_p.join(df_v).join(df_m)
    return df.groupby(df.index).last()


def descargar_global():
    """Devuelve el market cap total del mercado cripto en este momento (USD)."""
    data = get_con_reintento(f"{COINGECKO_BASE}/global")
    return float(data["data"]["total_market_cap"]["usd"])


def descargar_fear_greed(limit_dias):
    """Descarga Fear & Greed reciente."""
    data = get_con_reintento(ALTERNATIVE_BASE, {"limit": limit_dias, "format": "json"})
    df = pd.DataFrame(data["data"])
    df["date"] = pd.to_datetime(df["timestamp"].astype(int), unit="s").dt.normalize()
    df["fear_greed"] = df["value"].astype(int)
    df["FearGreed_Label"] = df["value_classification"]
    return df[["date", "fear_greed", "FearGreed_Label"]].set_index("date").sort_index()


print("Funciones definidas ✓")

Funciones definidas ✓


## 4. Salida temprana si no hay nada que actualizar

In [4]:
if dias_faltan <= 0:
    raise SystemExit("CSV ya al día. Detenido.")

if dias_faltan > MAX_DIAS_AUTO:
    raise SystemExit(f"Faltan {dias_faltan} días. Usa mergearcsv.ipynb.")

## 5. Descarga

In [5]:
dias_validos = ajustar_dias_coingecko(dias_faltan + 2)
print(f"Solicitando {dias_validos} días a CoinGecko (cubre {dias_faltan} + margen)\n")

print("→ Bitcoin...")
df_btc = descargar_market_chart("bitcoin", dias_validos)
df_btc.columns = ["btc_close", "btc_volume", "btc_mcap"]
print(f"  {len(df_btc)} filas")
time.sleep(PAUSA_API)

print("\n→ Ethereum...")
df_eth = descargar_market_chart("ethereum", dias_validos)
df_eth.columns = ["eth_close", "eth_volume", "eth_mcap"]
print(f"  {len(df_eth)} filas")
time.sleep(PAUSA_API)

print("\n→ Market cap total del mercado (CoinGecko /global, momento actual)...")
total_mcap_actual = descargar_global()
print(f"  Total mcap actual: ${total_mcap_actual:,.0f}")
time.sleep(PAUSA_API)

print("\n→ Fear & Greed...")
df_fg = descargar_fear_greed(limit_dias=dias_validos + 5)
print(f"  {len(df_fg)} filas")

Solicitando 7 días a CoinGecko (cubre 1 + margen)

→ Bitcoin...
  7 filas

→ Ethereum...
  7 filas

→ Market cap total del mercado (CoinGecko /global, momento actual)...
  Total mcap actual: $2,744,787,334,952

→ Fear & Greed...
  12 filas


## 6. Construcción del DataFrame nuevo

Para cada día nuevo (1-7 días), el `total_market_cap` se aproxima al actual.
El error es despreciable porque el mercado total no se mueve más del 1-2% en 1-2 días.

In [6]:
# 1. Combinar BTC + ETH
df_nuevo = df_btc.join(df_eth, how="outer")

# 2. Filtrar solo días estrictamente nuevos
df_nuevo = df_nuevo[df_nuevo.index > ultima_fecha].copy()
print(f"Días nuevos a añadir: {len(df_nuevo)}")

if df_nuevo.empty:
    raise SystemExit("No hay días nuevos tras filtrar.")

# 3. Calcular dominancias usando el total mcap actual
#    Para 1-2 días el error de aproximación es <0.5%
df_nuevo["btc_dominance"] = df_nuevo["btc_mcap"] / total_mcap_actual
df_nuevo["eth_dominance"] = df_nuevo["eth_mcap"] / total_mcap_actual
df_nuevo["alt_dominance"] = 1 - df_nuevo["btc_dominance"] - df_nuevo["eth_dominance"]

# 4. OHLC desde close (limitación API gratuita)
for activo in ("btc", "eth"):
    df_nuevo[f"{activo}_open"] = df_nuevo[f"{activo}_close"]
    df_nuevo[f"{activo}_high"] = df_nuevo[f"{activo}_close"]
    df_nuevo[f"{activo}_low"]  = df_nuevo[f"{activo}_close"]

# 5. Fear & Greed
df_nuevo = df_nuevo.join(df_fg, how="left")
df_nuevo["fear_greed"]      = df_nuevo["fear_greed"].ffill()
df_nuevo["FearGreed_Label"] = df_nuevo["FearGreed_Label"].ffill()

# 6. Inflation y fed_rate: forward-fill desde el último valor del CSV viejo
#    (los actualizas manualmente cuando salga el dato mensual)
df_nuevo["inflation"] = df_merged["inflation"].iloc[-1]
df_nuevo["fed_rate"]  = df_merged["fed_rate"].iloc[-1]

# 7. Reordenar columnas igual que el CSV viejo
df_nuevo = df_nuevo[df_merged.columns.tolist()]
df_nuevo.index.name = "date"

print(f"\nDataFrame nuevo listo: {df_nuevo.shape}")
print(f"Nulos: {df_nuevo.isna().sum().sum()}")
df_nuevo

Días nuevos a añadir: 1

DataFrame nuevo listo: (1, 19)
Nulos: 0


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-05-07,80158.778295,80158.778295,80158.778295,80158.778295,3.561218e+10,2295.944982,2295.944982,2295.944982,2295.944982,1.977681e+10,0.584852,0.10095,0.314198,47,Neutral,2.7,1.605295e+12,2.770854e+11,3.84


## 7. Backup y merge automático

Hace backup del CSV viejo en `data/csv/backups/df_merged_backup_AAAAMMDD.csv`,
concatena el nuevo, deduplica y guarda el actualizado.

In [7]:
# Backup
fecha_backup = hoy.strftime("%Y%m%d")
ruta_backup  = os.path.join(DIR_BACKUPS, f"df_merged_backup_{fecha_backup}.csv")
shutil.copy(RUTA_CSV, ruta_backup)
print(f"✓ Backup guardado: {ruta_backup}")

# Merge
df_actualizado = pd.concat([df_merged, df_nuevo])
df_actualizado = df_actualizado[~df_actualizado.index.duplicated(keep="last")]
df_actualizado.sort_index(inplace=True)

# Guardar
df_actualizado.to_csv(RUTA_CSV)

print(f"\n✓ CSV actualizado guardado: {RUTA_CSV}")
print(f"  Filas totales: {len(df_actualizado)} ({len(df_merged)} previas + {len(df_nuevo)} nuevas)")
print(f"  Rango: {df_actualizado.index.min().date()} → {df_actualizado.index.max().date()}")
print(f"\nÚltimas 3 filas:")
df_actualizado.tail(3)

✓ Backup guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\backups\df_merged_backup_20260507.csv

✓ CSV actualizado guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv
  Filas totales: 3018 (3017 previas + 1 nuevas)
  Rango: 2018-02-01 → 2026-05-07

Últimas 3 filas:


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-05-05,79823.885560,79823.885560,79823.885560,79823.885560,5.578048e+10,2345.995666,2345.995666,2345.995666,2345.995666,2.388095e+10,0.586547,0.103911,0.309542,50.0,Neutral,2.7,1.598024e+12,2.831011e+11,3.84
2026-05-06,81783.114270,81783.114270,81783.114270,81783.114270,4.576081e+10,2363.331772,2363.331772,2363.331772,2363.331772,2.181428e+10,0.592442,0.103089,0.304469,46.0,Fear,2.7,1.636674e+12,2.847918e+11,3.84
2026-05-07,80158.778295,80158.778295,80158.778295,80158.778295,3.561218e+10,2295.944982,2295.944982,2295.944982,2295.944982,1.977681e+10,0.584852,0.100950,0.314198,47.0,Neutral,2.7,1.605295e+12,2.770854e+11,3.84
